In [0]:
%sql
SHOW TABLES IN ipl.bronze

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

In [0]:
meta = spark.readStream.format('cloudFiles') \
.option('cloudFiles.schemaLocation','/Volumes/ipl/silver/schemalocation') \
.option('cloudFiles.format','parquet') \
.load('/Volumes/ipl/bronze/ipldata')

In [0]:
meta_selected = meta.select(col('meta.data_version').alias('dataVersion')
                            ,col('meta.created').alias('createdDate')
                            ,col('meta.revision').cast(IntegerType()).alias('revision')
                            ,col('_metadata.file_name').alias('fileName')
                            ,col('_metadata.file_path').alias('filePath')
                            )

In [0]:
meta_selected.writeStream.format('delta') \
.option('checkpointLocation','/Volumes/ipl/silver/checkpoint') \
.outputMode('append') \
.trigger(once=True) \
.toTable('ipl.silver.meta')